In [0]:
%run ../../config/utils

In [0]:
import argparse
from datetime import datetime
import os
import re

import numpy as np
import yaml

import sys
sys.path.append('..')
sys.path.append('../..')

from lib.utils import next_fiscal_week_end
import lib_trip_spend.python_general_utilities as util_func
import pyspark.sql.functions as f
import mlflow
import pandas as pd
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DoubleType, DateType

In [0]:
############## SET VARIABLES FROM CONFIG ###########################
config_path = './config/config.yml'

with open(config_path, "r") as stream:
    config = yaml.load(stream, Loader=yaml.FullLoader)

week = None if not dbutils.widgets.get("week") else dbutils.widgets.get("week")

In [0]:
############################ DEFINE GLOBAL VARIABLES ###############################
HIGH_FREQUENCY_VISITS_LAST_12_WEEKS = config["shared"][
    "high_frequency_visit_threshold"
]
LOW_FREQUENCY_VISITS_LAST_26_WEEKS = config["shared"][
    "low_frequency_visit_threshold"
]

if week is None:
    WEEKS_TO_PREDICT = config["predict"]["weeks_to_predict"]
else:
    WEEKS_TO_PREDICT = week
WEEKS = []

# The following code is necessary because the script predicts propensity for fiscal weekends only
WEEKS_TO_PREDICT = list(
    set([next_fiscal_week_end(w) for w in WEEKS_TO_PREDICT])
)

trip_model_uri = f"models:/{trip_propensity_model_catalog}@champion"
spend_model_uri = f"models:/{trip_spend_model_catalog}@champion"

In [0]:
########################### DEFINE FUNCTIONS ###############################
def split_features(features):
    categorical = [item for item in features if item.endswith("tmp")]
    continious = [item for item in features if item not in categorical]
    categorical = [item[:-4] for item in categorical]
    return categorical, continious


In [0]:
PRED_SCHEMA = StructType([
    StructField("MBRSHP_SID", LongType(), True),
    StructField("LATEST_PRI_SUPP_FHH_IND", IntegerType(), True),
    StructField("probability_making_a_trip", DoubleType(), True),
    StructField("predicted_make_trip", IntegerType(), True),
    StructField("predicted_spend", DoubleType(), True),
    StructField("FISCAL_WEEK_END", DateType(), True)
])

In [0]:
def save_to_volume_or_s3(panda_df_to_be_saved, recent_saturday_str):
    filename = f"trip_spend_predictions_{recent_saturday_str}.csv"

    # We will save to DBX Volume by default. If in prod we additionally move such file to s3
    run_name = config["shared"]["run_name"]

    file_path_in_volume = f'{tp_score_volume_path}/{run_name}/{filename}'  
    cnf_path_in_volume  = f'{tp_score_volume_path}/{run_name}/CONF/cnfg_predict.yml'  

    # Create dirs with run_name if it doesn't exist
    dbutils.fs.mkdirs(f'{tp_score_volume_path}/{run_name}')
    dbutils.fs.mkdirs(f'{tp_score_volume_path}/{run_name}/CONF')

    # Save config used to volume              
    dbutils.fs.cp(f"file:{os.path.abspath(config_path)}", cnf_path_in_volume) # we needed the full path to the config_file

    # Save tp_score to volume
    panda_df_to_be_saved.to_csv(file_path_in_volume, index=False, header=True)
    
    print(f"Output successfully saved to volume: {file_path_in_volume}.")



    # LL Note: we recommend EDW team to access this directly from UnityCatalog and delete this step and their related files/ resources

    if environment == 'prod':
        try:

            dbutils.fs.cp(file_path_in_volume, f'{tp_score_path}/{run_name}/{filename}') # just copy the 1st one so we can use 'move' in next step
            dbutils.fs.mv(file_path_in_volume, f'{tp_score_edw_path}/{filename}')
            dbutils.fs.cp(cnf_path_in_volume, f'{tp_score_path}/{run_name}/CONF/cnfg_predict.yml')
            print(f"Inference output saved to additional S3 locations ({tp_score_path}/{run_name}/ and {tp_score_edw_path}/{run_name}/) successfully.")

        except Exception as e:
            print(f"Error saving results to S3: {e}")

In [0]:
############################ RUN SCRIPT ##################################

trip_features = pd.read_csv(trip_feature_importance_path)
trip_features = trip_features["Feature_Name"].tolist()

spend_features = pd.read_csv(spend_feature_importance_path)
spend_features = spend_features["Feature_Name"].tolist()

categorical_features, continious_features = split_features(trip_features)

mlflow.set_registry_uri('databricks-uc')

for w in WEEKS_TO_PREDICT:
    start_time = w
    end_time = next_fiscal_week_end(w)
    data = spark.table(fs_customer_cube_full).filter((f.col("FISCAL_WEEK_END") >= start_time) & (f.col("FISCAL_WEEK_END") <= end_time)).toPandas()
    if len(data) == 0:
        continue
    print("-----start_time and end_time are: ------", start_time, end_time)
    print("predicting for week {}".format(w))

    # join with segment
    data.set_index(["MBRSHP_SID"], inplace=True, drop=False)
    data[continious_features] = data[continious_features].fillna(value=0)

    data = util_func.create_independent_variables(
        data,
        LOW_FREQUENCY_VISITS_LAST_26_WEEKS,
        HIGH_FREQUENCY_VISITS_LAST_12_WEEKS,
    )
    data = util_func.create_seasonality_fields(data)
    data = util_func.transform_categorical_features(data, categorical_features)

    print(
        "--------------------- transform numpy --------------------------------"
    )
    data_np_array_trip = data[trip_features].values
    data_np_array_trip = np.nan_to_num(data_np_array_trip)

    data_np_array_spend = data[spend_features].values
    data_np_array_spend = np.nan_to_num(data_np_array_spend)

    print("--------------------- load models --------------------------------")
    trip_model = mlflow.sklearn.load_model(trip_model_uri)
    spend_model = mlflow.sklearn.load_model(spend_model_uri)

    print(
        "--------------------- predict likelihood of trip --------------------------------"
    )
    data["probability_making_a_trip"] = trip_model.predict_proba(
        data_np_array_trip
    )[:, 1]
    data["predicted_make_trip"] = trip_model.predict(data_np_array_trip)
    print(
        "--------------------- predict spend --------------------------------"
    )
    data["predicted_spend"] = spend_model.predict(data_np_array_spend)
    data["predicted_spend"] = np.where(
        data["probability_making_a_trip"] <= 0.08, 0, data["predicted_spend"]
    )

    print("--------------------- write to table --------------------------------")

    recent_saturday_str     = datetime.strptime(w, "%Y-%m-%d")
    data["FISCAL_WEEK_END"] = recent_saturday_str.date()

    df_data = spark.createDataFrame(data[[
            "MBRSHP_SID",
            "LATEST_PRI_SUPP_FHH_IND",
            "probability_making_a_trip",
            "predicted_make_trip",
            "predicted_spend",
            "FISCAL_WEEK_END"
        ]], PRED_SCHEMA)
    
    df_data.write.mode("overwrite").option('replaceWhere', f"FISCAL_WEEK_END = '{w}'").saveAsTable(trip_spend_prediction)
    
    WEEKS_TO_PREDICT.remove(start_time)

    # Now lets save to volume or s3 too
    #-----------------------------------------------------
    data_as_legacy = data[[
            "MBRSHP_SID",
            "LATEST_PRI_SUPP_FHH_IND",
            "probability_making_a_trip",
            "predicted_make_trip",
        ]] # these columns were not saved in the original code

    save_to_volume_or_s3(panda_df_to_be_saved=data_as_legacy, recent_saturday_str=recent_saturday_str.strftime("%Y-%m-%d"))


if len(WEEKS_TO_PREDICT):
    print(
        "NOTE: There is no data for the following week(s): {}".format(
            WEEKS_TO_PREDICT
        )
    )